In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split,GridSearchCV
# from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score,f1_score,recall_score,roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

In [ ]:
df=pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'],errors='coerce')

In [ ]:
df["TotalCharges"].isnull().sum()

In [ ]:
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

In [ ]:
df["TotalCharges"].isnull().sum()

In [ ]:
df.drop("customerID",axis=1,inplace=True)

In [ ]:
df.head()

In [ ]:
df['Churn'].value_counts()

In [ ]:
df["Churn"]=df["Churn"].map({"Yes":1,"No":0})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
sns.boxplot(y=df['tenure'], ax=axes[0], color='lightblue')
axes[0].set_title('Tenure Outliers')
sns.boxplot(y=df['MonthlyCharges'], ax=axes[1], color='salmon')
axes[1].set_title('MonthlyCharges Outliers')
sns.boxplot(y=df['TotalCharges'], ax=axes[2], color='lightgreen')
axes[2].set_title('TotalCharges Outliers')
plt.tight_layout()
plt.show()

In [ ]:
# Q1: Churn distribution
sns.countplot(data=df, x='Churn', palette='Set2')
plt.title('Churn Distribution')
plt.show()
# Q2: Contract type vs Churn
sns.countplot(data=df, x='Contract',hue='Churn', palette='Set1')
plt.title('Contract Type vs Churn')
plt.show()
# Q3: Tenure distribution by Churn
sns.histplot(data=df, x='tenure',hue='Churn', bins=30, kde=True)
plt.title('Tenure vs Churn')
plt.show()
# Q4: MonthlyCharges vs Churn
sns.boxplot(data=df, x='Churn',y='MonthlyCharges', palette='coolwarm')
plt.title('Monthly Charges vs Churn')
plt.show()

In [ ]:
x=df.drop("Churn",axis=1)
y=df["Churn"]

In [ ]:
cat_column=x.select_dtypes(include='object').columns
num_column=x.select_dtypes(include="number").columns
# cat_column
num_column

In [ ]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,train_size=0.8,random_state=42,stratify=y)
xtrain,xtest,ytrain,ytest

In [ ]:
num_preprocessing=Pipeline(
    steps=[
    ("imputer",SimpleImputer(strategy="mean")),
    ("scaling",RobustScaler())
])
cat_preprocessing=Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="constant",fill_value="Unknown")),
    ("encoder",OneHotEncoder(sparse_output=False,handle_unknown="ignore"))
])

In [ ]:
smote=SMOTE(random_state=42)

In [ ]:
column_transform=ColumnTransformer(transformers=[
    ("num_transform",num_preprocessing,num_column),
    ("cat_transform",cat_preprocessing,cat_column)
])
column_transform

In [ ]:
xtrain_preprocessed = column_transform.fit_transform(xtrain)
xtrain_smote, ytrain_smote = smote.fit_resample(xtrain_preprocessed, ytrain)
print(pd.Series(ytrain_smote).value_counts())

In [ ]:
orginal_pipeline=Pipeline(steps=[
    ("complet_preprocessing",column_transform),
    ("model",LogisticRegression(random_state=42))
])
orginal_pipeline

In [ ]:
smote_model = Pipeline(steps=[
    ("model", LogisticRegression(random_state=42))

])
smote_model.fit(xtrain_smote, ytrain_smote)

In [ ]:
ytrain_predict = smote_model.predict(column_transform.transform(xtrain))
ytest_predict = smote_model.predict(column_transform.transform(xtest))
ytrain_predict
ytest_predict

In [ ]:
print("Accuracy:", accuracy_score(ytest, ytest_predict))
print(classification_report(ytest, ytest_predict))


In [ ]:
cm = confusion_matrix(ytest, ytest_predict)
sns.heatmap(cm, annot=True, fmt='d',cmap='Blues',xticklabels=['No Churn', 'Churn'],yticklabels=['No Churn', 'Churn'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
# 1. F1 Score (Churn class)
f1 = f1_score(ytest, ytest_predict, pos_label=1)
print("Churn F1 Score:", f1)  # Target: > 0.75
# 2. Recall (Churn class) — most important!
recall = recall_score(ytest, ytest_predict, pos_label=1)
print("Churn Recall:", recall)  # Target: > 0.75
# 3. ROC-AUC Score
y_prob = smote_model.predict_proba(column_transform.transform(xtest))[:, 1]
auc = roc_auc_score(ytest, y_prob)
print("ROC-AUC:", auc)  # Target: > 0.85

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier

models = {
    'Logistic Regression': LogisticRegression(
                           class_weight='balanced',random_state=42),
    'Random Forest': RandomForestClassifier(
                     n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(scale_pos_weight=2.77)
}

for name, m in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessing", column_transform),
        ("model", m)
    ])
    pipeline.fit(xtrain, ytrain)
    pred = pipeline.predict(xtest)
    f1 = f1_score(ytest,ytest_predict, pos_label=1)
    print(f'{name}: F1={f1:.3f}')

In [ ]:
# from sklearn.model_selection import StratifiedKFold, cross_val_score
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(orginal_pipeline, x, y,cv=cv, scoring='f1_weighted')
print("CV F1 Scores:", scores)
print("Mean F1:", scores.mean())

In [ ]:
#from sklearn.model_selection import GridSearchCV
param_grid = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__solver': ['lbfgs', 'liblinear']}
grid = GridSearchCV(
    orginal_pipeline, param_grid,
    cv=5, scoring='f1_weighted')
grid.fit(xtrain, ytrain)
print("Best params:", grid.best_params_)

In [ ]:
import joblib

# Save pipeline
joblib.dump(orginal_pipeline, 'churn_model.pkl')

# Streamlit app bannu
# → Customer data input pannuvom
# → Churn prediction show pannuvom
# → Resume-la "Deployed ML App" mention pannuvom!